# On-Policy Reverse-KL Distillation for Indian History (Qwen3-4B → Qwen3-1.7B)

| | |
|---|---|
| **Teacher** | `Qwen/Qwen3-4B`, bf16, frozen |
| **Student** | `Qwen/Qwen3-1.7B`, bf16, full fine-tune with gradient checkpointing |
| **Mode** | Non-thinking (`enable_thinking=False`) |
| **Method** | On-policy reverse KL: the student samples its own completions, the teacher scores them token-by-token, and the student minimises KL(student ‖ teacher) on the completion tokens |
| **Dataset** | `fully_tagged_dataset.jsonl`: 580 Indian-history essay questions, each tagged with a time-period bucket and a tonal/intent bucket (see `docs/dataset_taxonomy.md`) |
| **Hardware** | Single NVIDIA L4 24 GB (Google Colab) |

Pipeline: setup → load + split data → load models → prompt format → distillation loop → generate eval answers (student) → loss curve → baseline answers (un-distilled student) → save artifacts.
The student/base generations are compared downstream with an LLM-as-a-judge (GPT-5.4, run outside this notebook).

In [ ]:
# 1. Environment setup (Colab)

# Colab ships torchvision/torchaudio builds that crash `transformers` on import (CUDA mismatch).
# They are not needed here, so remove them rather than re-install torch.
!pip uninstall -y torchvision torchaudio
!pip install -U -q "transformers>=4.51.0" datasets accelerate tiktoken einops

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce fragmentation OOMs

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# 2. Load dataset

import json
from collections import Counter

DATA_PATH = "fully_tagged_dataset.jsonl"  # upload to the Colab runtime first

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"'{DATA_PATH}' not found. Upload it to the runtime via the file sidebar.")

dataset = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            dataset.append(json.loads(line))

assert len(dataset) > 0, "Dataset is empty"
assert all({"messages", "time_period_tag", "tonal_tag"} <= d.keys() for d in dataset), "Malformed entries"

print(f"Loaded {len(dataset)} questions from {DATA_PATH}")
print(f"Time-period distribution: {dict(sorted(Counter(d['time_period_tag'] for d in dataset).items()))}")
print(f"Tonal distribution:       {dict(sorted(Counter(d['tonal_tag'] for d in dataset).items()))}")

In [ ]:
# 3. Train / eval split (stratified on tonal x time-period bucket)

import random
from collections import defaultdict

random.seed(42)

TARGET_EVAL_SIZE = 80
eval_fraction = TARGET_EVAL_SIZE / len(dataset)

by_combo = defaultdict(list)
for item in dataset:
    by_combo[(item["tonal_tag"], item["time_period_tag"])].append(item)

train_data, eval_data = [], []
for combo, items in sorted(by_combo.items()):
    random.shuffle(items)
    # Proportional share per bucket, but at least 1 so every (tonal, period) combo is represented in eval
    n_eval = min(len(items), max(1, int(round(len(items) * eval_fraction))))
    eval_data.extend(items[:n_eval])
    train_data.extend(items[n_eval:])

random.shuffle(train_data)
random.shuffle(eval_data)

print(f"Total: {len(dataset)} | Train: {len(train_data)} | Eval: {len(eval_data)}")
print(f"Eval tonal breakdown:       {dict(Counter(d['tonal_tag'] for d in eval_data))}")
print(f"Eval time-period breakdown: {dict(Counter(d['time_period_tag'] for d in eval_data))}")

In [ ]:
# 4. Load teacher + student

import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

gc.collect()
torch.cuda.empty_cache()

TEACHER_ID = "Qwen/Qwen3-4B"
STUDENT_ID = "Qwen/Qwen3-1.7B"

# Both models are Qwen3 family and share a vocabulary, so one tokenizer serves both.
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for batched generation

teacher_tok = AutoTokenizer.from_pretrained(TEACHER_ID, trust_remote_code=True)
assert tokenizer.vocab_size == teacher_tok.vocab_size, \
    f"Vocab mismatch: student {tokenizer.vocab_size} vs teacher {teacher_tok.vocab_size}"
del teacher_tok

# Teacher: bf16, frozen. (4-bit loading via bitsandbytes was tried and dropped: the weight
# conversion OOMed on Colab, and bf16 Qwen3-4B is ~8 GB, which fits on an L4 alongside the student.)
print(f"Loading teacher {TEACHER_ID} (bf16)...")
teacher_model = AutoModelForCausalLM.from_pretrained(TEACHER_ID, torch_dtype=torch.bfloat16, device_map="auto")
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False
print(f"  Teacher loaded. VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

# Student: bf16, trainable, gradient checkpointing to keep activations small.
print(f"Loading student {STUDENT_ID} (bf16)...")
student_model = AutoModelForCausalLM.from_pretrained(STUDENT_ID, torch_dtype=torch.bfloat16, device_map="auto")
student_model.gradient_checkpointing_enable()
student_model.train()

print(f"\nTeacher params: {sum(p.numel() for p in teacher_model.parameters()) / 1e9:.2f}B")
print(f"Student params: {sum(p.numel() for p in student_model.parameters()) / 1e9:.2f}B")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
# 5. Prompt formatting (Qwen3 chat template, non-thinking mode)

SYSTEM_PROMPT = (
    "You are a knowledgeable historian specializing in Indian history. "
    "Provide balanced, well-reasoned, and nuanced answers to questions about Indian history. "
    "Do not be verbose, you have to be well reasoned yet concise. "
    "Acknowledge multiple perspectives where they exist. "
    "Be factually accurate and cite relevant historical context."
)

def format_prompt(question_text: str) -> str:
    """Render a question as a Qwen3 chat prompt with thinking disabled."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question_text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

sample_prompt = format_prompt(train_data[0]["messages"][0]["content"])
print(sample_prompt)
print(f"\nTokenized length: {len(tokenizer.encode(sample_prompt))} tokens")

In [ ]:
# 6. Hyperparameters and sampling settings (shared by training rollouts and eval generation)

MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8            # effective batch size = 8
EPOCHS = 1
LR = 2e-5
WARMUP_RATIO = 0.1
MAX_PROMPT_TOKENS = 384
MAX_NEW_TOKENS = 512            # student rollout length
MAX_SEQ_LEN = 896               # prompt + completion cap

# Qwen3 recommended non-thinking sampling parameters
GEN_KWARGS = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
)

def tokenize_prompts(questions):
    prompts = [format_prompt(q) for q in questions]
    return tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_PROMPT_TOKENS
    ).to("cuda:0")

In [ ]:
# 7. On-policy reverse-KL distillation loop

import math
import time
import torch.nn.functional as F
from transformers.optimization import Adafactor

optimizer = Adafactor(
    student_model.parameters(),
    lr=LR,
    scale_parameter=False,
    relative_step=False,
    warmup_init=False,
    weight_decay=0.01,
)

total_steps = math.ceil(len(train_data) / MICRO_BATCH_SIZE) * EPOCHS
opt_steps = total_steps // GRAD_ACCUM_STEPS
warmup_steps = int(opt_steps * WARMUP_RATIO)

def get_lr(step):
    """Linear warmup, then cosine decay to 0."""
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, opt_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

print("Training config:")
print(f"  Train samples:   {len(train_data)}")
print(f"  Micro batch:     {MICRO_BATCH_SIZE}  x  grad accum {GRAD_ACCUM_STEPS}  =  eff. batch {MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Epochs:          {EPOCHS}")
print(f"  Total steps:     {total_steps}  (optimizer steps: {opt_steps}, warmup: {warmup_steps})")
print(f"  Max rollout:     {MAX_NEW_TOKENS} tokens")
print(f"  Optimizer:       Adafactor")

def optimizer_step():
    torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

loss_history = []
global_step = 0
train_start_time = time.time()
total_gen_time = 0.0

for epoch in range(EPOCHS):
    random.shuffle(train_data)
    epoch_losses = []
    epoch_start = time.time()
    epoch_gen_time = 0.0

    for i in range(0, len(train_data), MICRO_BATCH_SIZE):
        batch = train_data[i : i + MICRO_BATCH_SIZE]

        # (a) Tokenise prompts
        prompt_inputs = tokenize_prompts([item["messages"][0]["content"] for item in batch])
        prompt_len = prompt_inputs["input_ids"].shape[1]

        # (b) On-policy rollout: the student samples its own completion
        student_model.eval()
        gen_start = time.time()
        with torch.no_grad():
            generated_ids = student_model.generate(**prompt_inputs, **GEN_KWARGS)
        gen_time = time.time() - gen_start
        epoch_gen_time += gen_time
        total_gen_time += gen_time
        student_model.train()

        if generated_ids.shape[1] > MAX_SEQ_LEN:
            generated_ids = generated_ids[:, :MAX_SEQ_LEN]
        if generated_ids.shape[1] - prompt_len < 2:
            continue  # degenerate rollout, skip

        attention_mask = (generated_ids != tokenizer.pad_token_id).long()

        # (c) Teacher (no grad) and student (grad) forward passes over the same rollout
        with torch.no_grad():
            teacher_logits = teacher_model(input_ids=generated_ids, attention_mask=attention_mask).logits
        student_logits = student_model(input_ids=generated_ids, attention_mask=attention_mask).logits

        # (d) Reverse KL on completion tokens only: KL(p_student || p_teacher), averaged over unmasked tokens.
        # Logit position t predicts token t+1, hence the [prompt_len-1 : -1] slice.
        s_logits = student_logits[:, prompt_len - 1 : -1, :]
        t_logits = teacher_logits[:, prompt_len - 1 : -1, :]
        comp_mask = attention_mask[:, prompt_len:].float()

        log_p_student = F.log_softmax(s_logits, dim=-1)
        log_p_teacher = F.log_softmax(t_logits, dim=-1)
        p_student = F.softmax(s_logits, dim=-1)

        per_token_kld = (p_student * (log_p_student - log_p_teacher)).sum(dim=-1)
        loss = (per_token_kld * comp_mask).sum() / comp_mask.sum().clamp(min=1)

        (loss / GRAD_ACCUM_STEPS).backward()

        epoch_losses.append(loss.item())
        loss_history.append(loss.item())
        global_step += 1

        # (e) Optimizer step every GRAD_ACCUM_STEPS micro-steps
        if global_step % GRAD_ACCUM_STEPS == 0:
            optimizer_step()

        if global_step % 20 == 0:
            avg_recent = sum(epoch_losses[-20:]) / len(epoch_losses[-20:])
            elapsed = time.time() - train_start_time
            eta_mins = (total_steps - global_step) * (elapsed / global_step) / 60
            print(
                f"Epoch {epoch+1}/{EPOCHS} | Step {global_step}/{total_steps} | "
                f"Reverse KLD: {loss.item():.4f} | Avg(20): {avg_recent:.4f} | "
                f"LR: {scheduler.get_last_lr()[0]:.2e} | VRAM: {torch.cuda.memory_allocated() / 1024**3:.1f}GB | "
                f"ETA: {eta_mins:.1f}m | GenTime/step: {total_gen_time / global_step:.2f}s"
            )

        del teacher_logits, student_logits, s_logits, t_logits, log_p_student, log_p_teacher, p_student
        del generated_ids, prompt_inputs, attention_mask
        torch.cuda.empty_cache()

    # Flush any leftover accumulated gradients at epoch end
    if global_step % GRAD_ACCUM_STEPS != 0:
        optimizer_step()

    epoch_avg = sum(epoch_losses) / len(epoch_losses)
    elapsed = time.time() - epoch_start
    print(f"\n{'=' * 70}")
    print(f"Epoch {epoch+1}/{EPOCHS} complete | Avg reverse KLD: {epoch_avg:.4f} | "
          f"Time: {elapsed / 60:.1f} min (generation: {epoch_gen_time / 60:.1f} min)")
    print(f"{'=' * 70}\n")

total_training_time = time.time() - train_start_time
print(f"Total training time:   {total_training_time / 60:.1f} min")
print(f"Total generation time: {total_gen_time / 60:.1f} min")

third = max(1, len(loss_history) // 3)
first_third = sum(loss_history[:third]) / third
last_third = sum(loss_history[-third:]) / third
print(f"\nLoss trend: first third avg={first_third:.4f} -> last third avg={last_third:.4f}")
print("Loss decreased." if last_third < first_third else "Loss did not decrease; check hyperparameters.")

In [ ]:
# 8. Eval generation helper (used for both the distilled student and the un-distilled baseline)

def generate_answers(model, eval_samples, answer_key, output_file, micro_batch=4):
    """Sample one answer per eval question with the given model and stream results to JSONL.

    Each line: {"question", <answer_key>, "time_period_tag", "tonal_tag"}.
    Written incrementally so a crash mid-way does not lose completed batches.
    """
    was_training = model.training
    model.eval()
    results = []
    start = time.time()
    total_batches = (len(eval_samples) + micro_batch - 1) // micro_batch
    print(f"Generating {len(eval_samples)} answers -> {output_file}")

    with torch.no_grad(), open(output_file, "w", encoding="utf-8") as f:
        for b, i in enumerate(range(0, len(eval_samples), micro_batch), start=1):
            batch = eval_samples[i : i + micro_batch]
            questions = [item["messages"][0]["content"] for item in batch]
            prompt_inputs = tokenize_prompts(questions)
            prompt_len = prompt_inputs["input_ids"].shape[1]

            generated_ids = model.generate(**prompt_inputs, **GEN_KWARGS, use_cache=True)
            completions = tokenizer.batch_decode(generated_ids[:, prompt_len:], skip_special_tokens=True)

            for item, q, c in zip(batch, questions, completions):
                entry = {
                    "question": q,
                    answer_key: c.strip(),
                    "time_period_tag": item.get("time_period_tag", ""),
                    "tonal_tag": item.get("tonal_tag", ""),
                }
                results.append(entry)
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

            del prompt_inputs, generated_ids
            torch.cuda.empty_cache()

            if b % 5 == 0 or b == total_batches:
                eta = (total_batches - b) * ((time.time() - start) / b) / 60
                print(f"  batch {b}/{total_batches} | ETA: {eta:.1f}m")

    print(f"Done in {(time.time() - start) / 60:.1f} min. Saved {len(results)} answers to {output_file}")
    if was_training:
        model.train()
    return results

In [ ]:
# 9. Distilled student: generate answers on the held-out eval set

eval_generations = generate_answers(
    student_model, eval_data, answer_key="student_answer", output_file="eval_student_generations.jsonl"
)

In [ ]:
# 10. Loss curve

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(loss_history, alpha=0.3, color="steelblue", label="Per-step")

window = 20
if len(loss_history) >= window:
    smoothed = [
        sum(loss_history[max(0, i - window) : i + 1]) / len(loss_history[max(0, i - window) : i + 1])
        for i in range(len(loss_history))
    ]
    plt.plot(smoothed, color="navy", linewidth=2, label=f"Smoothed ({window}-step)")

plt.xlabel("Step")
plt.ylabel("Reverse KLD")
plt.title("Training loss: on-policy reverse KLD")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 11. Save the distilled student + run metadata to Google Drive

from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/distilled_qwen3_1.7b_indian_history"
os.makedirs(SAVE_DIR, exist_ok=True)

student_model.eval()
student_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

metadata = {
    "teacher_model": TEACHER_ID,
    "student_model": STUDENT_ID,
    "method": "on-policy reverse KLD",
    "mode": "non-thinking (enable_thinking=False)",
    "dataset": DATA_PATH,
    "train_samples": len(train_data),
    "eval_samples": len(eval_data),
    "epochs": EPOCHS,
    "learning_rate": LR,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_seq_len": MAX_SEQ_LEN,
    "final_step_loss": loss_history[-1] if loss_history else None,
    "mean_loss_last_20_steps": sum(loss_history[-20:]) / len(loss_history[-20:]) if loss_history else None,
}
with open(f"{SAVE_DIR}/training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model, tokenizer and metadata to {SAVE_DIR}")
print(f'Reload with: AutoModelForCausalLM.from_pretrained("{SAVE_DIR}")')

In [ ]:
# 12. Baseline: answers from the un-distilled Qwen3-1.7B on the same eval set

# Free the training models first so the base model fits alongside nothing else.
for name in ("student_model", "teacher_model"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(STUDENT_ID, torch_dtype=torch.bfloat16, device_map="auto")
base_model.eval()

base_generations = generate_answers(
    base_model, eval_data, answer_key="base_answer", output_file="eval_base_generations.jsonl"
)

In [ ]:
# 13. Copy eval outputs to Google Drive (input to the GPT-5.4 LLM-as-a-judge comparison, run separately)

import shutil

for fname in ("eval_student_generations.jsonl", "eval_base_generations.jsonl", "training_curves.png"):
    shutil.copy(fname, f"/content/drive/MyDrive/{fname}")
    print(f"Copied {fname} to Drive")